# Time dependent boundary condition

## Imports

In [2]:
import zoomy_core

In [5]:
# | code-fold: true
# | code-summary: "Load packages"
# | output: false

import os
import numpy as np
import jax
from jax import numpy as jnp
import pytest
from types import SimpleNamespace
from sympy import cos, pi, Piecewise, sin, Matrix
import sympy

from zoomy_jax.fvm.solver_jax import HyperbolicSolver
from zoomy_core.fvm.ode import RK1
# import zoomy_core.fvm.reconstruction as recon
import zoomy_core.fvm.timestepping as timestepping
import zoomy_core.fvm.flux as flux
import zoomy_core.fvm.nonconservative_flux as nc_flux
from zoomy_core.model.basemodel import eigenvalue_dict_to_matrix
from zoomy_core.model.boundary_conditions import BoundaryCondition
from zoomy_core.model.models.basisfunctions import Basisfunction, Legendre_shifted, Legendre_DN, Chebyshevu
from zoomy_core.model.models.basismatrices import Basismatrices
from zoomy_core.model.models.shallow_moments import ShallowMoments
from zoomy_core.model.models.shallow_moments_topo import ShallowMomentsTopoNumerical,ShallowMomentsTopo
import zoomy_core.misc.io as io

import zoomy_core.model.initial_conditions as IC
import zoomy_core.model.boundary_conditions as BC
import zoomy_core.misc.io as io
from zoomy_core.mesh.mesh import compute_derivatives
from zoomy_tests.swashes import plots_paper

from zoomy_core.misc.misc import Zstruct, Settings


import zoomy_core.mesh.mesh as petscMesh
import zoomy_core.postprocessing.postprocessing as postprocessing
# from zoomy_core.mesh.mesh import convert_mesh_to_jax
import argparse
from zoomy_core.model.models.basisfunctions import Legendre_shifted, Basisfunction
from zoomy_core.model.models.basismatrices import Basismatrices

import matplotlib.pyplot as plt


## Model

In [6]:
level = 2
basis = Legendre_shifted

settings = Settings(
    name="sme",
    output=Zstruct(
        directory=f"outputs/timedependent_bc", filename="sme", snapshots=100, clean_directory=True
    ),
)


In [11]:
testbasis = basis(level)
Z = np.linspace(0, 1, 100)
U = lambda z: 1-(1-z)**8
U_a = U(Z)
coefs = testbasis.project_onto_basis(U_a)
U_b = testbasis.reconstruct_velocity_profile(np.array(coefs), N = Z.shape[0])
plt.plot(U_b, Z)
plt.plot(U_a, Z)
inflow_profile_coefs = coefs

SympifyError: SympifyError: array([0.        , 0.01010101, 0.02020202, 0.03030303, 0.04040404,
       0.05050505, 0.06060606, 0.07070707, 0.08080808, 0.09090909,
       0.1010101 , 0.11111111, 0.12121212, 0.13131313, 0.14141414,
       0.15151515, 0.16161616, 0.17171717, 0.18181818, 0.19191919,
       0.2020202 , 0.21212121, 0.22222222, 0.23232323, 0.24242424,
       0.25252525, 0.26262626, 0.27272727, 0.28282828, 0.29292929,
       0.3030303 , 0.31313131, 0.32323232, 0.33333333, 0.34343434,
       0.35353535, 0.36363636, 0.37373737, 0.38383838, 0.39393939,
       0.4040404 , 0.41414141, 0.42424242, 0.43434343, 0.44444444,
       0.45454545, 0.46464646, 0.47474747, 0.48484848, 0.49494949,
       0.50505051, 0.51515152, 0.52525253, 0.53535354, 0.54545455,
       0.55555556, 0.56565657, 0.57575758, 0.58585859, 0.5959596 ,
       0.60606061, 0.61616162, 0.62626263, 0.63636364, 0.64646465,
       0.65656566, 0.66666667, 0.67676768, 0.68686869, 0.6969697 ,
       0.70707071, 0.71717172, 0.7

In [4]:

f_h = lambda t: 0.2 * cos(2 * 3.14 * t/2) + 1.
u_mean = 0.5

inflow_dict = {
    0: lambda t, x, dx, q, qaux, p, n: 0.0,
    1: lambda t, x, dx, q, qaux, p, n: f_h(t),
    2: lambda t, x, dx, q, qaux, p, n: sympy.sqrt(f_h(t) * 9.81) * q[1],
}
inflow_dict.update({2 + i: lambda t, x, dx, q, qaux, p, n:  0.0 for i in range(level)})

bcs = BC.BoundaryConditions(
    [
        BC.Lambda(tag="left", prescribe_fields=inflow_dict),
        BC.Extrapolation(tag="right"),
    ]
)

def custom_ic(x):
    Q = np.zeros(3 + level, dtype=float)
    Q[0] = 0.05 * x[0]
    Q[1] = 1.
    return Q


ic = IC.UserFunction(custom_ic)

class MyModel(ShallowMomentsTopo):   
    def source(self):   
        out = Matrix([0 for i in range(self.n_variables)])
        out += self.slip_mod()
        # out += self.newtonian_turbulent_algebraic()
        # out += self.regress_against_power_profile()
        return out
               
model = MyModel(
    dimension=1,
    level = level,
    boundary_conditions=bcs,
    aux_variables = 3 + level,
    parameters=Zstruct(C=30., lamda=0.001, rho=1000., kappa=0.41, l_bl=0.01, l_turb=0.05, nu=1e-6, c_slipmod=1., r_pp=10.),
    basisfunctions=basis,
    initial_conditions=ic
)

main_dir = os.getenv("ZOOMY_DIR")
mesh = petscMesh.Mesh.create_1d([0.0, 10.0], 400)



In [5]:
class SMESolver(HyperbolicSolver):
    def update_qaux(self, Q, Qaux, Qold, Qauxold, mesh, model, parameters, time, dt):
        Qaux = Qaux.at[0].set(compute_derivatives(Q[0], mesh, derivatives_multi_index=[[1, 0]])[:, 0])
        Qaux = Qaux.at[1].set(compute_derivatives(Q[1], mesh, derivatives_multi_index=[[1, 0]])[:, 0])

        for i in range(2, Q.shape[0]):
            Qaux = Qaux.at[i].set(compute_derivatives(Q[i]/Q[1], mesh, derivatives_multi_index=[[1, 0]])[:, 0])
        return Qaux


solver = SMESolver(settings=settings, time_end=30.0, compute_dt=timestepping.adaptive(CFL=0.9))

## Solver

In [6]:
Qnew, Qaux = solver.solve(mesh, model)

2025-09-24 12:56:55.520 | INFO     | library.core.fvm.solver_jax:log_callback_hyperbolic:44 - iteration: 10, time: 0.064186, dt: 0.006288, next write at time: 0.303030
2025-09-24 12:56:55.528 | INFO     | library.core.fvm.solver_jax:log_callback_hyperbolic:44 - iteration: 20, time: 0.127076, dt: 0.006290, next write at time: 0.303030
2025-09-24 12:56:55.536 | INFO     | library.core.fvm.solver_jax:log_callback_hyperbolic:44 - iteration: 30, time: 0.189985, dt: 0.006292, next write at time: 0.303030
2025-09-24 12:56:55.542 | INFO     | library.core.fvm.solver_jax:log_callback_hyperbolic:44 - iteration: 40, time: 0.252909, dt: 0.006293, next write at time: 0.303030
2025-09-24 12:56:55.553 | INFO     | library.core.fvm.solver_jax:log_callback_hyperbolic:44 - iteration: 50, time: 0.315849, dt: 0.006295, next write at time: 0.606061
2025-09-24 12:56:55.560 | INFO     | library.core.fvm.solver_jax:log_callback_hyperbolic:44 - iteration: 60, time: 0.378804, dt: 0.006296, next write at time: 0

## Visualization

In [7]:
io.generate_vtk(os.path.join(settings.output.directory, f"{settings.name}.h5"))
postprocessing.vtk_project_2d_to_3d(model, settings, Nz=50, filename='out_3d')

2025-09-24 12:57:00.114 | INFO     | library.postprocessing.postprocessing:vtk_project_2d_to_3d:62 - Converted snapshot 0/100
2025-09-24 12:57:00.131 | INFO     | library.postprocessing.postprocessing:vtk_project_2d_to_3d:62 - Converted snapshot 1/100
2025-09-24 12:57:00.144 | INFO     | library.postprocessing.postprocessing:vtk_project_2d_to_3d:62 - Converted snapshot 2/100
2025-09-24 12:57:00.159 | INFO     | library.postprocessing.postprocessing:vtk_project_2d_to_3d:62 - Converted snapshot 3/100
2025-09-24 12:57:00.172 | INFO     | library.postprocessing.postprocessing:vtk_project_2d_to_3d:62 - Converted snapshot 4/100
2025-09-24 12:57:00.185 | INFO     | library.postprocessing.postprocessing:vtk_project_2d_to_3d:62 - Converted snapshot 5/100
2025-09-24 12:57:00.198 | INFO     | library.postprocessing.postprocessing:vtk_project_2d_to_3d:62 - Converted snapshot 6/100
2025-09-24 12:57:00.211 | INFO     | library.postprocessing.postprocessing:vtk_project_2d_to_3d:62 - Converted snapsho